In [3]:
import sys
print(sys.executable)

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully!")
print("Pandas version:", pd.__version__)

c:\Users\User\Documents\Coding\data analyst\messi-dash\venv\Scripts\python.exe
Libraries loaded successfully!
Pandas version: 3.0.5


In [4]:
# Sesuaikan path sesuai lokasi file Anda
df = pd.read_csv("../data/messi dataset/messi_goals_assists_2008_2026.csv")

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Shape: (531, 16)

First 5 rows:


,date,season,competition,stage,team,opponent,venue,stadium,city,country,goals,assists,club_or_country,career_goals,career_assists,career_goal_contributions
0,2005-05-01,2004-05,La Liga,League Matchday,FC Barcelona,Albacete,Home,Camp Nou,Barcelona,Spain,1,0,FC Barcelona,1,0,1
1,2005-11-27,2005-06,La Liga,League Matchday,FC Barcelona,Racing Santander,Home,Camp Nou,Barcelona,Spain,1,0,FC Barcelona,2,0,2
2,2006-01-15,2005-06,La Liga,League Matchday,FC Barcelona,Athletic Bilbao,Home,Camp Nou,Barcelona,Spain,1,0,FC Barcelona,3,0,3
3,2006-01-22,2005-06,La Liga,League Matchday,FC Barcelona,Alaves,Home,Camp Nou,Barcelona,Spain,1,0,FC Barcelona,4,0,4
4,2006-01-29,2005-06,La Liga,League Matchday,FC Barcelona,Mallorca,Away,Son Moix,Palma,Spain,2,0,FC Barcelona,6,0,6


In [5]:
print("="*60)
print("DATASET OVERVIEW")
print("="*60)

print(f"\nTotal rows     : {df.shape[0]}")
print(f"Total columns  : {df.shape[1]}")

print("\nColumn names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nBasic Statistics (Goals & Assists):")
print(df[['goals', 'assists', 'career_goals', 'career_assists', 'career_goal_contributions']].describe())

DATASET OVERVIEW

Total rows     : 531
Total columns  : 16

Column names:
['date', 'season', 'competition', 'stage', 'team', 'opponent', 'venue', 'stadium', 'city', 'country', 'goals', 'assists', 'club_or_country', 'career_goals', 'career_assists', 'career_goal_contributions']

Data Types:
date                           str
season                         str
competition                    str
stage                          str
team                           str
opponent                       str
venue                          str
stadium                        str
city                           str
country                        str
goals                        int64
assists                      int64
club_or_country                str
career_goals                 int64
career_assists               int64
career_goal_contributions    int64
dtype: object

Missing Values:
date                          0
season                        0
competition                   0
stage                 

In [6]:
# 1. Copy dataframe agar data asli aman
df_clean = df.copy()

# 2. Convert date ke datetime
df_clean['date'] = pd.to_datetime(df_clean['date'])

# 3. Buat kolom waktu tambahan
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month
df_clean['year_month'] = df_clean['date'].dt.to_period('M').astype(str)
df_clean['weekday'] = df_clean['date'].dt.day_name()

# 4. Standarisasi nama klub / negara
def standardize_club(club):
    if club == 'FC Barcelona':
        return 'Barcelona'
    elif club == 'Paris Saint-Germain' or club == 'PSG':
        return 'PSG'
    elif 'Inter Miami' in str(club):
        return 'Inter Miami'
    elif club == 'Argentina':
        return 'Argentina'
    else:
        return club

df_clean['club'] = df_clean['club_or_country'].apply(standardize_club)

# 5. Flag International vs Club
df_clean['is_international'] = df_clean['club'] == 'Argentina'

# 6. Venue Type (Home / Away / Neutral)
df_clean['venue_type'] = df_clean['venue'].str.strip().str.title()

# 7. Stage Category (untuk analisis lebih mudah)
def categorize_stage(stage):
    stage = str(stage).lower()
    if 'final' in stage:
        return 'Final'
    elif any(x in stage for x in ['semi', 'sf', 'quarter', 'qf', 'r16', 'round of 16', 'r32']):
        return 'Knockout'
    elif 'group' in stage:
        return 'Group Stage'
    elif 'league' in stage or 'matchday' in stage:
        return 'League'
    else:
        return 'Other'

df_clean['stage_category'] = df_clean['stage'].apply(categorize_stage)

# 8. Goal Contribution per match
df_clean['goal_contribution'] = df_clean['goals'] + df_clean['assists']

# 9. Urutkan berdasarkan tanggal (penting untuk running total)
df_clean = df_clean.sort_values('date').reset_index(drop=True)

# 10. Validasi running totals (opsional tapi bagus)
print("Validasi Running Totals:")
print(f"Max Career Goals     : {df_clean['career_goals'].max()}")
print(f"Max Career Assists   : {df_clean['career_assists'].max()}")
print(f"Max Career G+A       : {df_clean['career_goal_contributions'].max()}")
print(f"Total Goals (sum)    : {df_clean['goals'].sum()}")
print(f"Total Assists (sum)  : {df_clean['assists'].sum()}")

print("\nCleaning & Feature Engineering selesai!")
print("Shape setelah cleaning:", df_clean.shape)
df_clean.head()

Validasi Running Totals:
Max Career Goals     : 675
Max Career Assists   : 304
Max Career G+A       : 979
Total Goals (sum)    : 675
Total Assists (sum)  : 304

Cleaning & Feature Engineering selesai!
Shape setelah cleaning: (531, 25)


,date,season,competition,stage,team,opponent,venue,stadium,city,country,goals,assists,club_or_country,career_goals,career_assists,career_goal_contributions,year,month,year_month,weekday,club,is_international,venue_type,stage_category,goal_contribution
0,2005-05-01,2004-05,La Liga,League Matchday,FC Barcelona,Albacete,Home,Camp Nou,Barcelona,Spain,1,0,FC Barcelona,1,0,1,2005,5,2005-05,Sunday,Barcelona,False,Home,League,1
1,2005-11-27,2005-06,La Liga,League Matchday,FC Barcelona,Racing Santander,Home,Camp Nou,Barcelona,Spain,1,0,FC Barcelona,2,0,2,2005,11,2005-11,Sunday,Barcelona,False,Home,League,1
2,2006-01-15,2005-06,La Liga,League Matchday,FC Barcelona,Athletic Bilbao,Home,Camp Nou,Barcelona,Spain,1,0,FC Barcelona,3,0,3,2006,1,2006-01,Sunday,Barcelona,False,Home,League,1
3,2006-01-22,2005-06,La Liga,League Matchday,FC Barcelona,Alaves,Home,Camp Nou,Barcelona,Spain,1,0,FC Barcelona,4,0,4,2006,1,2006-01,Sunday,Barcelona,False,Home,League,1
4,2006-01-29,2005-06,La Liga,League Matchday,FC Barcelona,Mallorca,Away,Son Moix,Palma,Spain,2,0,FC Barcelona,6,0,6,2006,1,2006-01,Sunday,Barcelona,False,Away,League,2


In [7]:
print("Unique Clubs:")
print(df_clean['club'].value_counts())

print("\nVenue Type:")
print(df_clean['venue_type'].value_counts())

print("\nStage Category:")
print(df_clean['stage_category'].value_counts())

print("\nInternational vs Club:")
print(df_clean['is_international'].value_counts())

Unique Clubs:
club
Barcelona      367
Inter Miami     74
Argentina       48
PSG             42
Name: count, dtype: int64

Venue Type:
venue_type
Home       278
Away       216
Neutral     37
Name: count, dtype: int64

Stage Category:
stage_category
League         426
Knockout        43
Group Stage     29
Other           26
Final            7
Name: count, dtype: int64

International vs Club:
is_international
False    483
True      48
Name: count, dtype: int64


In [8]:
# Membuat Milestone Tracker
milestones = [100, 200, 300, 400, 500, 600, 650, 675]

milestone_data = []

for m in milestones:
    # Cari baris pertama kali career_goals mencapai atau melebihi milestone
    row = df_clean[df_clean['career_goals'] >= m].iloc[0]
    
    milestone_data.append({
        'Milestone': f"{m} Goals",
        'Date': row['date'].strftime('%Y-%m-%d'),
        'Match': f"{row['team']} vs {row['opponent']}",
        'Competition': row['competition'],
        'Club': row['club'],
        'Goals in Match': row['goals'],
        'Career Goals at that time': row['career_goals']
    })

milestone_df = pd.DataFrame(milestone_data)
print("Messi Career Goals Milestone Tracker")
print("="*80)
milestone_df

Messi Career Goals Milestone Tracker


,Milestone,Date,Match,Competition,Club,Goals in Match,Career Goals at that time
0,100 Goals,2010-11-20,FC Barcelona vs UD Almeria,La Liga,Barcelona,3,101
1,200 Goals,2013-01-27,FC Barcelona vs CA Osasuna,La Liga,Barcelona,4,202
2,300 Goals,2016-02-17,FC Barcelona vs Sporting Gijon,La Liga,Barcelona,2,301
3,400 Goals,2019-02-02,FC Barcelona vs Valencia,La Liga,Barcelona,2,400
4,500 Goals,2021-06-28,Argentina vs Bolivia,Copa America,Argentina,2,501
5,600 Goals,2024-10-15,Argentina vs Bolivia,CONMEBOL World Cup Qualifiers,Argentina,3,600
6,650 Goals,2025-11-23,Inter Miami CF vs FC Cincinnati,MLS Cup Playoffs,Inter Miami,1,650
7,675 Goals,2026-08-05,Inter Miami CF vs Atletico San Luis,Leagues Cup,Inter Miami,2,675


In [9]:
print("="*60)
print("QUICK CAREER SUMMARY")
print("="*60)

print(f"\nTotal Matches with G/A     : {len(df_clean)}")
print(f"Total Goals                 : {df_clean['goals'].sum()}")
print(f"Total Assists               : {df_clean['assists'].sum()}")
print(f"Total Goal Contributions    : {df_clean['goal_contribution'].sum()}")

print(f"\nAverage Goals per Match     : {df_clean['goals'].mean():.2f}")
print(f"Average Assists per Match   : {df_clean['assists'].mean():.2f}")
print(f"Average G+A per Match       : {df_clean['goal_contribution'].mean():.2f}")

print("\n--- By Club ---")
club_summary = df_clean.groupby('club').agg({
    'goals': 'sum',
    'assists': 'sum',
    'goal_contribution': 'sum',
    'date': 'count'
}).rename(columns={'date': 'matches'}).sort_values('goal_contribution', ascending=False)

club_summary['G+A per Match'] = (club_summary['goal_contribution'] / club_summary['matches']).round(2)
print(club_summary)

QUICK CAREER SUMMARY

Total Matches with G/A     : 531
Total Goals                 : 675
Total Assists               : 304
Total Goal Contributions    : 979

Average Goals per Match     : 1.27
Average Assists per Match   : 0.57
Average G+A per Match       : 1.84

--- By Club ---
             goals  assists  goal_contribution  matches  G+A per Match
club                                                                  
Barcelona      491      194                685      367           1.87
Inter Miami     92       50                142       74           1.92
Argentina       60       26                 86       48           1.79
PSG             32       34                 66       42           1.57


In [10]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [11]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_clean['date'], 
    y=df_clean['career_goals'],
    mode='lines',
    name='Career Goals',
    line=dict(color='#A50044', width=2.5)
))

fig.add_trace(go.Scatter(
    x=df_clean['date'], 
    y=df_clean['career_assists'],
    mode='lines',
    name='Career Assists',
    line=dict(color='#004D98', width=2.5)
))

fig.add_trace(go.Scatter(
    x=df_clean['date'], 
    y=df_clean['career_goal_contributions'],
    mode='lines',
    name='Goal Contributions',
    line=dict(color='#FFD700', width=2.5)
))

fig.update_layout(
    title='Lionel Messi - Career Cumulative Goals, Assists & Contributions (2005–2026)',
    xaxis_title='Date',
    yaxis_title='Cumulative Count',
    hovermode='x unified',
    template='plotly_white',
    height=550,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.show()

In [12]:
club_stats = df_clean.groupby('club').agg({
    'goals': 'sum',
    'assists': 'sum',
    'goal_contribution': 'sum'
}).reset_index()

fig = px.bar(
    club_stats,
    x='club',
    y=['goals', 'assists'],
    barmode='group',
    title='Total Goals & Assists by Club / National Team',
    labels={'value': 'Count', 'variable': 'Type'},
    color_discrete_map={'goals': '#A50044', 'assists': '#004D98'},
    text_auto=True
)

fig.update_layout(template='plotly_white', height=500)
fig.show()

In [13]:
venue_stats = df_clean.groupby('venue_type').agg({
    'goals': 'sum',
    'assists': 'sum',
    'goal_contribution': 'sum',
    'date': 'count'
}).rename(columns={'date': 'matches'}).reset_index()

venue_stats['G+A per Match'] = (venue_stats['goal_contribution'] / venue_stats['matches']).round(2)

fig = px.bar(
    venue_stats,
    x='venue_type',
    y='goal_contribution',
    color='venue_type',
    title='Goal Contributions by Venue Type',
    text='goal_contribution',
    color_discrete_sequence=['#A50044', '#004D98', '#FFD700']
)

fig.update_layout(template='plotly_white', height=450, showlegend=False)
fig.show()

print("\nDetail Venue Performance:")
print(venue_stats)


Detail Venue Performance:
  venue_type  goals  assists  goal_contribution  matches  G+A per Match
0       Away    258      139                397      216           1.84
1       Home    375      145                520      278           1.87
2    Neutral     42       20                 62       37           1.68
